# GSS Demographic Simulation — Activation Polarization

Instead of simulating politicians, we feed actual GSS respondent demographics to the LLM
and capture activation polarization on public/private issue questions.

**Approach:**
- Each prompt gives the respondent's demographic profile (randomized field order)
- Survey question appended at the end
- Extract head activations → PCA (15 PCs) → Mahalanobis distance (mean + median centroids)
- Correlate LLM activation polarization with GSS survey polarization

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import torch
import gc
import time
import warnings
from datetime import datetime
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings('ignore')

from model_utils import load_model, extract_heads_batched, get_model_info
from run_gss_pca import (
    compute_mahalanobis_pca,
    compute_all_head_metrics_pca,
)

/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# Configuration
MODEL_PATH = '/project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct'
MODEL_NAME = 'Llama-3.1-8B-Instruct'
GSS_DATA_PATH = '/project/jevans/maxzhuyt/gss-depth/gss_2021_2024.csv'
PCA_DIMS = [15]
BATCH_SIZE = 128
MAX_LENGTH = 512  # longer prompts with all demographic fields
OUTPUT_DIR = Path('llm_results')
OUTPUT_DIR.mkdir(exist_ok=True)
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

SYSTEM_MSG = 'You are simulating the views of an American.'

# Load all demographic fields from CSV
_demo_csv = pd.read_csv('gss_question_lists/gss_demographic_variables.csv')
DEMO_FIELDS = list(_demo_csv['VariableName'])
DEMO_LABELS = dict(zip(_demo_csv['VariableName'], _demo_csv['ConciseDescription']))

# polviews is not in the demographic CSV but should be included
if 'polviews' not in DEMO_FIELDS:
    DEMO_FIELDS.append('polviews')
    DEMO_LABELS['polviews'] = 'Political views'

print(f'Demographic fields: {len(DEMO_FIELDS)}')
del _demo_csv

# Filtering thresholds for polarization data
MIN_DEM = 100
MIN_REP = 100
MIN_TOTAL = 200

# Topic exclusion lists
EXCLUDE_PUBLIC = {'hubbywk1', 'racdif1', 'racdif2', 'racdif3', 'racdif4', 'workwhts', 'wlthwhts', 'intlwhts'}
EXCLUDE_PRIVATE = {'reborn', 'marwht', 'helpful', 'helpfulnv', 'helpfulv'}

Demographic fields: 104


## 1. Load model

In [3]:
print(f'CUDA available: {torch.cuda.is_available()}')
model, tokenizer = load_model(MODEL_PATH)
model_info = get_model_info(model)
print(f'Layers: {model_info["num_layers"]}, Heads: {model_info["num_heads"]}, Head dim: {model_info["head_dim"]}')

CUDA available: True
Loading model from: /project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.51s/it]

Layers: 32, Heads: 32, Head dim: 128


## 2. Load GSS data and build code mappings

In [4]:
# Load GSS survey data
df_gss = pd.read_csv(GSS_DATA_PATH, low_memory=False)
print(f'GSS data: {df_gss.shape[0]} respondents, {df_gss.shape[1]} variables')
print(f'Years: {sorted(df_gss["year"].unique())}')

# Build code-to-label mappings from Stata files
print('\nLoading value labels from Stata files...')
df_num_24 = pd.read_stata('/project/jevans/maxzhuyt/gss-depth/GSS2024.dta', convert_categoricals=False)
df_cat_24 = pd.read_stata('/project/jevans/maxzhuyt/gss-depth/GSS2024.dta', convert_categoricals=True)

df_num_22 = pd.read_stata('/project/jevans/maxzhuyt/gss-depth/GSS2022.dta', convert_categoricals=False)
reader_22 = pd.io.stata.StataReader('/project/jevans/maxzhuyt/gss-depth/GSS2022.dta')
vl_22 = reader_22.value_labels()
var_to_lbl_22 = dict(zip(reader_22._varlist, reader_22._lbllist))

def build_code_map(var_name, max_code=100000):
    """Extract code→label mapping for a variable from Stata files."""
    # Try 2024 first
    if var_name in df_num_24.columns:
        mapping = {}
        for n, c in zip(df_num_24[var_name], df_cat_24[var_name]):
            if pd.notna(n) and pd.notna(c) and int(n) < max_code:
                mapping[int(n)] = str(c).strip()
        if mapping:
            return mapping
    # Fall back to 2022
    if var_name in df_num_22.columns:
        lbl = var_to_lbl_22.get(var_name, '')
        if lbl and lbl in vl_22:
            return {int(k): str(v).strip() for k, v in vl_22[lbl].items() if int(k) < max_code}
    return {}

# Build code maps for demographic fields + polviews
code_maps = {}
for field in DEMO_FIELDS:
    code_maps[field] = build_code_map(field)
    n = len(code_maps[field])
    sample = dict(list(code_maps[field].items())[:3]) if n > 0 else {}
    print(f'  {field}: {n} codes, e.g. {sample}')

# Clean up Stata dataframes
del df_num_24, df_cat_24, df_num_22
gc.collect()

GSS data: 11066 respondents, 1771 variables
Years: [np.int64(2021), np.int64(2022), np.int64(2024)]

Loading value labels from Stata files...
  adults: 6 codes, e.g. {2: '2.0', 1: '1 adult in household', 4: '4.0'}
  age: 72 codes, e.g. {33: '33.0', 64: '64.0', 69: '69.0'}
  agekdbrn: 29 codes, e.g. {21: '21.0', 23: '23.0', 28: '28.0'}
  babies: 3 codes, e.g. {1: '1', 0: '0', 2: '2 or more'}
  born: 2 codes, e.g. {1: 'yes', 2: 'no'}
  childs: 9 codes, e.g. {2: '2.0', 0: '0.0', 3: '3.0'}
  degree: 5 codes, e.g. {3: "bachelor's", 4: 'graduate', 2: 'associate/junior college'}
  denom: 7 codes, e.g. {1: 'baptist', 3: 'lutheran', 6: 'other denomination'}
  denom16: 7 codes, e.g. {1: 'baptist', 3: 'lutheran', 4: 'presbyterian'}
  dipged: 3 codes, e.g. {1: 'high school diploma', 2: 'ged', 3: 'other'}
  divorce: 2 codes, e.g. {2: 'no', 1: 'yes'}
  earnrs: 4 codes, e.g. {2: '2.0', 1: '1.0', 0: '0.0'}
  educ: 21 codes, e.g. {16: '4 years of college', 14: '2 years of college', 12: '12th grade'}
  

0

In [ ]:
# Filter to Democrats and Republicans
# partyid: 0=Strong D, 1=Not strong D, 2=Ind near D,
#          3=Independent, 4=Ind near R, 5=Not strong R, 6=Strong R
df_gss['party_code'] = np.nan
df_gss.loc[df_gss['partyid'].isin([0, 1, 2]), 'party_code'] = 100  # Democrat
df_gss.loc[df_gss['partyid'].isin([4, 5, 6]), 'party_code'] = 200  # Republican

df_dr = df_gss[df_gss['party_code'].notna()].copy()
print(f'D/R respondents: {len(df_dr)} ({(df_dr["party_code"]==100).sum()} D, {(df_dr["party_code"]==200).sum()} R)')

## 3. Build demographic profiles

In [6]:
def precompute_demo_parts(df, code_maps, demo_fields, demo_labels):
    """
    Pre-compute demographic key-value strings for each respondent.
    Returns dict mapping row index → list of 'Label: value' strings.
    """
    all_parts = {}
    for idx in df.index:
        row = df.loc[idx]
        parts = []
        for field in demo_fields:
            val = row.get(field)
            if pd.isna(val):
                continue
            code = int(val)
            label = demo_labels[field]
            if field in code_maps and code in code_maps[field]:
                text = code_maps[field][code]
            else:
                text = str(code)
            parts.append(f'{label}: {text}')
        all_parts[idx] = parts
    return all_parts


# Pre-compute for all D/R respondents (done once, reused across topics)
demo_parts = precompute_demo_parts(df_dr, code_maps, DEMO_FIELDS, DEMO_LABELS)
print(f'Pre-computed demographic parts for {len(demo_parts)} respondents')

# Show sample profiles
rng = np.random.default_rng(42)
print('\nSample profiles:')
for idx in list(df_dr.index)[:3]:
    parts = demo_parts[idx].copy()
    rng.shuffle(parts)
    profile = '. '.join(parts) + '.'
    party = 'D' if df_dr.loc[idx, 'party_code'] == 100 else 'R'
    print(f'  [{party}] {profile}')
    print()

Pre-computed demographic parts for 6192 respondents

Sample profiles:
  [D] Number of siblings: 1.0. Preteens in household: 0. Urban/rural classification: other urban (counties having towns of 10,000 or more). Hours worked last week: 16.0. Health: fair. Region at 16: northeast. City type: a suburb of a medium size central city. Mother's education: bachelor's. City size: 24. Supervisor's supervisor: yes. Father's industry: computer systems design and related services. Mother's occupation: personal care and service workers, all other. Race: white. Residence type at 16: in a large city (over 250,000). Self-employed: someone else. Father's education: bachelor's. Industry: furniture and home furnishings stores. Region: northeast. Teens in household: 0. Full-time or part-time: part-time. Mother's industry: other personal services. Weeks worked last year: 50.0. Partnership status: i don't have a steady partner. Father's occupation: designers. Family structure at 16: both own parents. Children

## 4. Load topics and polarization data

In [7]:
# Load topic descriptions
pub_topics = pd.read_csv('gss_question_lists/public_issues.csv')
priv_topics = pd.read_csv('gss_question_lists/private_life.csv')

# Map variable name → survey question text
topic_questions = {}
for _, row in pd.concat([pub_topics, priv_topics]).iterrows():
    topic_questions[row['Variable']] = row['SurveyQuestion']

# Load polarization data
pol_pub = pd.read_csv('public_issues_polarization.csv')
pol_priv = pd.read_csv('private_life_polarization.csv')

# Filter by minimum respondent counts
pol_pub = pol_pub[(pol_pub['n_dem'] >= MIN_DEM) & (pol_pub['n_rep'] >= MIN_REP) & (pol_pub['n_total'] >= MIN_TOTAL)]
pol_priv = pol_priv[(pol_priv['n_dem'] >= MIN_DEM) & (pol_priv['n_rep'] >= MIN_REP) & (pol_priv['n_total'] >= MIN_TOTAL)]

# Build category config
categories = {
    'public_issues': {
        'topics': {row['Variable']: row['SurveyQuestion'] 
                   for _, row in pub_topics.iterrows() 
                   if row['Variable'] in set(pol_pub['variable'])},
        'polarization': pol_pub,
    },
    'private_life': {
        'topics': {row['Variable']: row['SurveyQuestion'] 
                   for _, row in priv_topics.iterrows() 
                   if row['Variable'] in set(pol_priv['variable'])},
        'polarization': pol_priv,
    },
}

for cat, cfg in categories.items():
    print(f'{cat}: {len(cfg["topics"])} topics, {len(cfg["polarization"])} polarization entries')

public_issues: 134 topics, 134 polarization entries
private_life: 75 topics, 87 polarization entries


## 5. Run activation extraction per topic

In [8]:
def is_valid_response(val, max_valid_code=97):
    """Check if a response code is valid (not missing, not metadata)."""
    if pd.isna(val):
        return False
    try:
        code = int(float(val))
        # GSS uses high codes for metadata: 0/8/9, 98/99, 997/998/999, etc.
        # But many valid codes are in the range of the scale.
        # The safest approach: just check not NaN and not super high
        return code < 1000000  # exclude Stata metadata codes only
    except (ValueError, TypeError):
        return False


def run_topic(model, tokenizer, topic_name, survey_question, df_respondents,
              demo_parts, pca_dims, batch_size, max_length):
    """Run activation extraction and PCA analysis for a single topic."""
    t0 = time.time()
    
    # Filter to respondents with valid answers for this topic
    if topic_name not in df_respondents.columns:
        return None
    
    valid_mask = df_respondents[topic_name].apply(is_valid_response)
    df_valid = df_respondents[valid_mask]
    
    if len(df_valid) < 20:
        print(f'    {topic_name}: SKIP (only {len(df_valid)} valid respondents)', flush=True)
        return None
    
    n_dem = (df_valid['party_code'] == 100).sum()
    n_rep = (df_valid['party_code'] == 200).sum()
    if n_dem < 10 or n_rep < 10:
        print(f'    {topic_name}: SKIP (D={n_dem}, R={n_rep})', flush=True)
        return None
    
    # Build prompts using pre-computed demographic parts
    rng = np.random.default_rng(hash(topic_name) % (2**32))
    prompts = []
    for idx in df_valid.index:
        parts = demo_parts[idx].copy()
        rng.shuffle(parts)
        profile = '. '.join(parts) + '.'
        prompts.append(f'{profile}\n\nSurvey question: {survey_question}')
    
    labels = df_valid['party_code'].values.astype(int)
    
    # Extract activations
    X_heads = extract_heads_batched(
        model, tokenizer, prompts, SYSTEM_MSG,
        batch_size=batch_size, max_length=max_length
    )
    
    # Compute PCA Mahalanobis for mean and median
    results = {'Topic': topic_name, 'n_valid': len(df_valid), 'n_dem': n_dem, 'n_rep': n_rep}
    
    for centroid_method in ['mean', 'median']:
        suffix = '' if centroid_method == 'mean' else '_median'
        for n_comp in pca_dims:
            grid = compute_all_head_metrics_pca(
                X_heads, labels,
                group_values=(100, 200),
                n_components=n_comp,
                centroid_method=centroid_method
            )
            results[f'Avg_Mahal_PCA{n_comp}{suffix}'] = np.mean(grid)
            results[f'Max_Mahal_PCA{n_comp}{suffix}'] = np.max(grid)
    
    del X_heads
    
    elapsed = time.time() - t0
    d = pca_dims[0]
    print(f'    {topic_name}: mean={results[f"Avg_Mahal_PCA{d}"]:.3f}, '
          f'median={results[f"Avg_Mahal_PCA{d}_median"]:.3f} '
          f'(n={len(df_valid)}, D={n_dem}, R={n_rep}, {elapsed:.1f}s)', flush=True)
    
    return results

In [ ]:
# Run analysis for all topics
category_results = {}

for cat_name, cat_cfg in categories.items():
    print(f'\n{"="*70}')
    print(f'ANALYZING: {cat_name.upper()}')
    print(f'{"="*70}')
    print(f'  Topics: {len(cat_cfg["topics"])}')
    print()
    
    results = []
    for idx, (topic_name, survey_q) in enumerate(cat_cfg['topics'].items()):
        try:
            result = run_topic(
                model, tokenizer, topic_name, survey_q, df_dr,
                demo_parts,
                pca_dims=PCA_DIMS, batch_size=BATCH_SIZE, max_length=MAX_LENGTH
            )
            if result is not None:
                result['category'] = cat_name
                results.append(result)
            
            if (idx + 1) % 20 == 0:
                gc.collect()
                torch.cuda.empty_cache()
                print(f'    [Memory cleanup at {idx+1}/{len(cat_cfg["topics"])}]')
        
        except Exception as e:
            print(f'    ERROR on {topic_name}: {e}', flush=True)
            gc.collect()
            torch.cuda.empty_cache()
    
    df_result = pd.DataFrame(results)
    category_results[cat_name] = df_result
    
    out_path = OUTPUT_DIR / f'df_demo_sim_{cat_name}_{MODEL_NAME}_{TIMESTAMP}.pkl'
    df_result.to_pickle(out_path)
    print(f'  Saved: {out_path}')


ANALYZING: PUBLIC_ISSUES
  Topics: 134

  > Extracting with Batch Size 128...


/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for y

    abdefect: mean=0.450, median=0.399 (n=2379, D=1371, R=1008, 234.1s)
  > Extracting with Batch Size 128...
    abhlth: mean=0.422, median=0.382 (n=2396, D=1388, R=1008, 202.8s)
  > Extracting with Batch Size 128...
    abnomore: mean=0.431, median=0.386 (n=2368, D=1351, R=1017, 199.0s)
  > Extracting with Batch Size 128...
    abpoor: mean=0.470, median=0.422 (n=2389, D=1371, R=1018, 200.8s)
  > Extracting with Batch Size 128...
    abrape: mean=0.420, median=0.367 (n=2388, D=1378, R=1010, 200.9s)
  > Extracting with Batch Size 128...
    absingle: mean=0.476, median=0.425 (n=2383, D=1371, R=1012, 200.6s)
  > Extracting with Batch Size 128...
    abhelp1: mean=0.486, median=0.447 (n=657, D=391, R=266, 56.2s)
  > Extracting with Batch Size 128...
    abhelp2: mean=0.526, median=0.489 (n=657, D=392, R=265, 56.3s)
  > Extracting with Batch Size 128...
    abhelp3: mean=0.468, median=0.452 (n=657, D=391, R=266, 56.3s)
  > Extracting with Batch Size 128...
    abhelp4: mean=0.494, median

/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for y

    affrmact: mean=0.451, median=0.424 (n=3951, D=2266, R=1685, 342.7s)
  > Extracting with Batch Size 128...


/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for y

    discaff: mean=0.398, median=0.344 (n=6033, D=3497, R=2536, 518.7s)
  > Extracting with Batch Size 128...
    discaffm: mean=0.484, median=0.419 (n=2017, D=1174, R=843, 172.9s)
  > Extracting with Batch Size 128...
    discaffw: mean=0.431, median=0.391 (n=1098, D=601, R=497, 93.8s)
  > Extracting with Batch Size 128...


/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for y

    fehire: mean=0.506, median=0.472 (n=4105, D=2370, R=1735, 356.7s)
    [Memory cleanup at 20/134]
  > Extracting with Batch Size 128...


## 6. Correlation analysis

In [ ]:
print('=' * 70)
print('CORRELATION ANALYSIS')
print('=' * 70)

all_correlations = []

for cat_name, cat_cfg in categories.items():
    df_llm = category_results[cat_name]
    df_pol = cat_cfg['polarization']
    
    df_merged = df_llm.merge(
        df_pol[['variable', 'polarization', 'area']].rename(
            columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
        ),
        on='Topic', how='inner'
    )
    
    print(f'\n[{cat_name.upper()}]')
    for method in ['mean', 'median']:
        suffix = '' if method == 'mean' else '_median'
        print(f'  Centroid: {method}')
        for d in PCA_DIMS:
            col = f'Avg_Mahal_PCA{d}{suffix}'
            if col in df_merged.columns:
                r_p = df_merged[col].corr(df_merged['GSS_Polarization'], method='pearson')
                r_s = df_merged[col].corr(df_merged['GSS_Polarization'], method='spearman')
                print(f'    PCA-{d:2d}: r={r_p:.4f}, rho={r_s:.4f} (n={len(df_merged)})')
                all_correlations.append({
                    'category': cat_name,
                    'pca_dim': d,
                    'centroid_method': method,
                    'n_topics': len(df_merged),
                    'pearson': r_p,
                    'spearman': r_s,
                    'df_merged': df_merged,
                    'llm_col': col,
                })

# Save correlation summary
df_corr = pd.DataFrame([{k: v for k, v in c.items() if k != 'df_merged'} for c in all_correlations])
corr_path = OUTPUT_DIR / f'demo_sim_correlations_{MODEL_NAME}_{TIMESTAMP}.csv'
df_corr.to_csv(corr_path, index=False)
print(f'\nSaved: {corr_path}')

## 7. Filtered results (topic exclusion)

In [ ]:
print('=' * 70)
print('FILTERED RESULTS (Topic Exclusion)')
print('=' * 70)

excluded_sets = {
    'public_issues': EXCLUDE_PUBLIC,
    'private_life': EXCLUDE_PRIVATE,
}

filtered_results = []

for corr in all_correlations:
    cat_name = corr['category']
    df_merged = corr['df_merged'].copy()
    llm_col = corr['llm_col']
    
    excluded = excluded_sets[cat_name]
    df_filtered = df_merged[~df_merged['Topic'].isin(excluded)].reset_index(drop=True)
    
    r_p = df_filtered[llm_col].corr(df_filtered['GSS_Polarization'], method='pearson')
    r_s = df_filtered[llm_col].corr(df_filtered['GSS_Polarization'], method='spearman')
    
    filtered_results.append({
        'Category': cat_name,
        'PCA_Dim': corr['pca_dim'],
        'Centroid_Method': corr['centroid_method'],
        'N_Original': len(df_merged),
        'N_Filtered': len(df_filtered),
        'Pearson_Original': corr['pearson'],
        'Pearson_Filtered': r_p,
        'Spearman_Original': corr['spearman'],
        'Spearman_Filtered': r_s,
    })

df_filt = pd.DataFrame(filtered_results)
print(f'\nExcluded (public): {sorted(EXCLUDE_PUBLIC)}')
print(f'Excluded (private): {sorted(EXCLUDE_PRIVATE)}')
print(f'\n{df_filt.to_string()}')

filt_path = OUTPUT_DIR / f'demo_sim_filtered_{MODEL_NAME}_{TIMESTAMP}.csv'
df_filt.to_csv(filt_path, index=False)
print(f'\nSaved: {filt_path}')

In [ ]:
# Save combined results
df_combined = pd.concat(list(category_results.values()), ignore_index=True)
df_combined['model'] = MODEL_NAME

combined_path = OUTPUT_DIR / f'df_demo_sim_combined_{MODEL_NAME}_{TIMESTAMP}.csv'
df_combined.to_csv(combined_path, index=False)
df_combined.to_pickle(combined_path.with_suffix('.pkl'))

print(f'Saved: {combined_path}')
print(f'\nTotal topics analyzed: {len(df_combined)}')
print(df_combined[['Topic', 'category', 'n_valid', 'n_dem', 'n_rep', f'Avg_Mahal_PCA{PCA_DIMS[0]}', f'Avg_Mahal_PCA{PCA_DIMS[0]}_median']].to_string())